# 2026-11-04-bt-FeSe-ob002-MvsT-Fw-Bokeh

## Imports, Aesthetics and Bokeh Setup

In [6]:
# ========================================
# CODE IMPORTS
# ========================================
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import os
import statistics
from scipy.interpolate import UnivariateSpline

# ========================================
# PLOT IMPORTS
# ========================================


from bokeh.layouts import column
from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.layouts import column
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, TabPanel, Tabs, Span
from bokeh.palettes import Category20_20, Category10_10

output_notebook()


# ========================================
# FUNCTION IMPORTS
# ========================================
sys.path.insert(1, './..')
import functions
from functions import read_squid_file
# determine_critical_temp

Loading BokehJS ...

## Data Reading

In [7]:
print("FIELD WARMING")
print("="*40)

from pathlib import Path
runs_fw = {}
pathlist = Path(
    "../../data/2026/10/FeSe_bo001_MvsT_FW_VariousFields"
).glob("**/*.dat")
for path in sorted(pathlist):
    run_number = path.stem[-5:]
    run_number = int(run_number) - 1
    runs_fw[run_number] = read_squid_file(str(path))
for run_number, data in runs_fw.items():
    print(f"Run {run_number}")

FIELD WARMING
Run 0
Run 1
Run 2
Run 3
Run 4


## Plotting a Raw data Line in Bokeh

In [8]:

p = figure(width=800,
           height=450,
           title = "261004 - FeSe ob002 FW",
           title_location = "above",
           x_axis_label="Temperature (K)",
           y_axis_label="DC Moment Fixed Ctr (emu)",
           )


for i in range(len(runs_fw)):
    avg_field = np.mean(runs_fw[i]["Magnetic Field (Oe)"])
    p.line(runs_fw[i]["Temperature (K)"],
           runs_fw[i]["DC Moment Fixed Ctr (emu)"],
           line_color=Category20_20[2*i],
           legend_label=f"Run {i}: Field {avg_field:.1f} (Oe)")


p.add_tools(HoverTool(tooltips=[("Temperature (K)", "@x{0.00}"),
                                ("DC Moment (emu)", "@y{0.000e}")]))

p.add_layout(p.legend[0], 'right')
p.legend[0].click_policy = "hide"
p.legend[0].title = "Field Runs"

show(p)

## Add fits and plot this
- note this code was heavily generated by gemini 3.1, it took my above code and various prompts, once functionallity was there i added comments and changed style to my desired 
- at a later date this will be refactored optimally by myself (human)

In [9]:


output_notebook()

# ========================================
# FUNCTION IMPORTS
# ========================================
sys.path.insert(1, './..')
try:
    from functions import read_squid_file
except ImportError:
    def read_squid_file(filepath):
        return pd.read_csv(filepath, skiprows=31)

print("FIELD WARMING - FITTED & DERIVATIVE")
print("="*40)

runs_fw = {}
pathlist = sorted(Path("../../data/2026/10/FeSe_bo001_MvsT_FW_VariousFields").glob("**/*.dat"))

for path in pathlist:
    run_number = int(path.stem[-5:]) - 1
    runs_fw[run_number] = read_squid_file(str(path))

tabs = []

for i, (run_number, data) in enumerate(runs_fw.items()):
    print(f"Processing Run {run_number}")
    
    df = data.dropna(subset=["Temperature (K)", "DC Moment Fixed Ctr (emu)"]).sort_values("Temperature (K)")

    t = df["Temperature (K)"].values
    m = df["DC Moment Fixed Ctr (emu)"].values
    avg_field = np.mean(df["Magnetic Field (Oe)"])
    
    color = Category20_20[(2 * i) % 20]
    color_off = Category20_20[(2 * i + 1) % 20]
    
    m_scale = np.max(np.abs(m))
    m_norm = m / m_scale
    
    spline = UnivariateSpline(t, m_norm, s=0.005)
    t_fit = np.linspace(t.min(), t.max(), 300)
    
    m_fit = spline(t_fit) * m_scale
    dm_dt = spline.derivative()(t_fit) * m_scale
    
    # 1. Tc Midpoint (Max Derivative)
    max_idx = np.argmax(dm_dt)
    tc_mid = t_fit[max_idx]
    m_mid = m_fit[max_idx]
    
    # 2. Tc Onset Calculation
    # Walk towards higher T from the peak until derivative drops to 5% of the max
    onset_threshold = 0.05 * dm_dt[max_idx]
    idx_onset = max_idx
    while idx_onset < len(dm_dt) - 1 and dm_dt[idx_onset] > onset_threshold:
        idx_onset += 1
        
    tc_onset = t_fit[idx_onset]
    m_onset = m_fit[idx_onset]
    
    # Half-max mask for the dashed transition line
    half_max = dm_dt[max_idx] / 2
    trans_mask = dm_dt > half_max
    
    p1 = figure(
        width=800, height=400,
        title=f"Run {run_number} | {avg_field:.0f} Oe (Tc Onset: {tc_onset:.2f} K, Mid: {tc_mid:.2f} K)",
        x_axis_label="", y_axis_label="DC Moment (emu)"
    )
    p2 = figure(
        width=800, height=250,
        x_axis_label="Temperature (K)", y_axis_label="dM/dT (emu/K)",
        x_range=p1.x_range
    )

    hover1 = HoverTool(tooltips=[("T (K)", "@x{0.00}"), ("M (emu)", "@y{0.000e}")])
    hover2 = HoverTool(tooltips=[("T (K)", "@x{0.00}"), ("dM/dT", "@y{0.000e}")])
    p1.add_tools(hover1)
    p2.add_tools(hover2)

    # Plot M(T)
    p1.scatter(t, m, size=3, color=color, alpha=0.25)
    fit_line = p1.line(t_fit, m_fit, line_width=2, line_color=color)
    p1.line(t_fit[trans_mask], m_fit[trans_mask], line_width=3, line_color=color_off, line_dash="dashed")
    
    # Markers for M(T)
    p1.scatter([tc_mid], [m_mid], size=10, marker="circle", color=color_off, line_color="black")
    p1.scatter([tc_onset], [m_onset], size=8, marker="square", color="red", line_color="black")
    
    # Plot dM/dT
    deriv_line = p2.line(t_fit, dm_dt, line_width=2, line_color=color)
    p2.line(t_fit[trans_mask], dm_dt[trans_mask], line_width=3, line_color=color_off, line_dash="dashed")
    
    # Markers for dM/dT
    p2.scatter([tc_mid], [dm_dt[max_idx]], size=10, marker="circle", color=color_off, line_color="black")
    p2.scatter([tc_onset], [dm_dt[idx_onset]], size=8, marker="square", color="red", line_color="black")

    # Add vertical lines to tie the two plots together visually
    mid_span = Span(location=tc_mid, dimension='height', line_color=color_off, line_dash='dashed', line_width=1)
    onset_span = Span(location=tc_onset, dimension='height', line_color='red', line_dash='dotted', line_width=1.5)
    
    p1.add_layout(mid_span)
    p1.add_layout(onset_span)
    p2.add_layout(mid_span)
    p2.add_layout(onset_span)

    hover1.renderers = [fit_line]
    hover2.renderers = [deriv_line]

    panel = TabPanel(child=column(p1, p2), title=f"{avg_field:.0f} Oe")
    tabs.append(panel)

show(Tabs(tabs=tabs))

Loading BokehJS ...

FIELD WARMING - FITTED & DERIVATIVE
Processing Run 0
Processing Run 1
Processing Run 2
Processing Run 3
Processing Run 4


In [10]:
fields, tc_mids, tc_onsets = [], [], []

# Re-extract parameters to keep the cell independent
for data in runs_fw.values():
    df = data.dropna(subset=["Temperature (K)", "DC Moment Fixed Ctr (emu)"]).sort_values("Temperature (K)")
    t, m = df["Temperature (K)"].values, df["DC Moment Fixed Ctr (emu)"].values
    
    m_norm = m / np.max(np.abs(m))
    spline = UnivariateSpline(t, m_norm, s=0.005)
    t_fit = np.linspace(t.min(), t.max(), 300)
    dm_dt = spline.derivative()(t_fit)
    
    max_idx = np.argmax(dm_dt)
    fields.append(np.mean(df["Magnetic Field (Oe)"]))
    tc_mids.append(t_fit[max_idx])
    
    onset_threshold = 0.05 * dm_dt[max_idx]
    idx_onset = max_idx
    while idx_onset < len(dm_dt) - 1 and dm_dt[idx_onset] > onset_threshold:
        idx_onset += 1
    tc_onsets.append(t_fit[idx_onset])

fields = np.array(fields)

# Calculate T^2 for the empirical parabolic law analysis
tc_mids_sq = np.array(tc_mids)**2
tc_onsets_sq = np.array(tc_onsets)**2

# Linear fits: T^2 = -[Tc(0)^2 / Hc(0)] * H + Tc(0)^2
fit_mid = np.polyfit(fields, tc_mids_sq, 1)
fit_onset = np.polyfit(fields, tc_onsets_sq, 1)
h_range = np.array([fields.min(), fields.max()])

# Extract physical parameters (y-int = Tc(0)^2, slope = -Tc(0)^2 / Hc(0))
tc_0_mid = np.sqrt(fit_mid[1])
hc_0_mid = -fit_mid[1] / fit_mid[0]

tc_0_onset = np.sqrt(fit_onset[1])
hc_0_onset = -fit_onset[1] / fit_onset[0]

p_sum2 = figure(
    width=800, height=400,
    title="Empirical Parabolic Law: T_c² vs. Applied Field",
    x_axis_label="Magnetic Field H (Oe)", y_axis_label="T_c² (K²)"
)
p_sum2.add_tools(HoverTool(tooltips=[("H (Oe)", "@x{0.0}"), ("T_c² (K²)", "@y{0.000}")]))

# Midpoint Data & Fit
p_sum2.scatter(fields, tc_mids_sq, size=10, marker="circle", color="navy", line_color="black", legend_label="Tc² Midpoint")
p_sum2.line(h_range, np.polyval(fit_mid, h_range), line_width=2, color="navy", line_dash="dashed",
           legend_label=f"Midpoint Fit (Tc(0) = {tc_0_mid:.2f} K, Hc(0) = {hc_0_mid:.0f} Oe)")

# Onset Data & Fit
p_sum2.scatter(fields, tc_onsets_sq, size=8, marker="square", color="red", line_color="black", legend_label="Tc² Onset")
p_sum2.line(h_range, np.polyval(fit_onset, h_range), line_width=2, color="red", line_dash="dashed",
           legend_label=f"Onset Fit (Tc(0) = {tc_0_onset:.2f} K, Hc(0) = {hc_0_onset:.0f} Oe)")

p_sum2.legend.click_policy = "hide"
p_sum2.legend.location = "top_right"

show(p_sum2)